# City-Hour Extension: Forecasting, Diwali, COVID Lockdown, PCA, City Comparison
Continues `city_hour_analysis.ipynb` (reads its cleaned output `data/processed/city_hour_clean.csv.gz`).
1 Next-hour & next-day forecasting · 2 Diwali effect · 3 COVID-19 lockdown effect · 4 PCA vs raw features · 5 City comparison · 6 SQLite export.

In [1]:
import pandas as pd, numpy as np, matplotlib, os, sqlite3, warnings
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid'); RS=42
FIG='../outputs/figures/city_hour/'; RES='../outputs/results/'
def save(n): plt.tight_layout(); plt.savefig(FIG+n,dpi=130); plt.show()
df=pd.read_csv('../data/processed/city_hour_clean.csv.gz',parse_dates=['Datetime'])
POLL=['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene']
print(df.shape)

(650264, 29)


## 1. Next-hour and next-day AQI forecasting
Unlike the first notebook (predicting the *current* AQI), here we use **only information available now or earlier** to predict the **future** AQI (t+1 hour and t+24 hours).
* Each city is put on a complete hourly grid first, so a lag of 24 rows is *exactly* 24 hours (missing hours become NaN instead of silently shifting).
* Rows with no target or no current AQI are dropped; the target is never imputed.
* Train 2015-2019, test 2020 (no shuffling). Baseline = **persistence** ("future AQI = AQI now"), which is the standard honest baseline for forecasting.

In [2]:
parts=[]
for city,g in df.groupby('City'):
    g=g.set_index('Datetime').sort_index()
    g=g[~g.index.duplicated()].reindex(pd.date_range(g.index.min(),g.index.max(),freq='h'))
    g['City']=city; parts.append(g)
d=pd.concat(parts); d.index.name='Datetime'; d=d.reset_index()
d['Hour']=d.Datetime.dt.hour; d['Month']=d.Datetime.dt.month; d['Year']=d.Datetime.dt.year; d['Weekday']=d.Datetime.dt.dayofweek
d['hour_sin']=np.sin(2*np.pi*d.Hour/24); d['hour_cos']=np.cos(2*np.pi*d.Hour/24); d['is_winter']=d.Month.isin([11,12,1,2]).astype(int)
g=d.groupby('City')
for L in (1,3,6,24): d[f'AQI_lag{L}']=g.AQI.shift(L)
d['AQI_roll24']=g.AQI.transform(lambda s:s.rolling(24,min_periods=12).mean())
for h in (1,24): d[f'target_{h}h']=g.AQI.shift(-h)
FEAT=['AQI','AQI_lag1','AQI_lag3','AQI_lag6','AQI_lag24','AQI_roll24','PM2.5','PM10','NO2','CO','O3','Hour','Month','Weekday','hour_sin','hour_cos','is_winter']
d=pd.concat([d,pd.get_dummies(d.City,prefix='city').astype(int)],axis=1); FEAT+= [c for c in d if c.startswith('city_')]
rows=[];store={}
for h in (1,24):
    t=f'target_{h}h'; k=d.dropna(subset=['AQI',t]).copy()
    tr=k[k.Year<=2019]; te=k[k.Year==2020]; trs=tr.sample(min(100000,len(tr)),random_state=RS)
    imp=SimpleImputer(strategy='median').fit(trs[FEAT]); A,B=imp.transform(trs[FEAT]),imp.transform(te[FEAT])
    res={'Persistence (baseline)':te.AQI.values,
         'Linear Regression':LinearRegression().fit(A,trs[t]).predict(B)}
    rf=RandomForestRegressor(n_estimators=60,max_depth=18,min_samples_leaf=5,n_jobs=-1,random_state=RS).fit(A,trs[t]); res['Random Forest']=rf.predict(B)
    for n,p in res.items(): rows.append([f'{h}h ahead',n,mean_absolute_error(te[t],p),np.sqrt(mean_squared_error(te[t],p)),r2_score(te[t],p)])
    store[h]=(te,res,rf)
fc=pd.DataFrame(rows,columns=['Horizon','Model','MAE','RMSE','R2']).round(3); fc.to_csv(RES+'city_hour_forecast_results.csv',index=False); fc

,Horizon,Model,MAE,RMSE,R2
0,1h ahead,Persistence (baseline),2.733,16.169,0.968
1,1h ahead,Linear Regression,4.049,14.863,0.973
2,1h ahead,Random Forest,2.982,14.095,0.976
3,24h ahead,Persistence (baseline),23.838,56.428,0.608
4,24h ahead,Linear Regression,27.870,54.821,0.630
5,24h ahead,Random Forest,25.622,54.610,0.632


In [3]:
# Plot: actual vs predicted for Delhi, 14 days of 2020 (24h ahead) + feature importance
te,res,rf=store[24]; mask=(te.City=='Delhi').values
dd=te[mask].set_index('Datetime'); dd['pred']=res['Random Forest'][mask]; dd['persist']=res['Persistence (baseline)'][mask]
w=dd.loc['2020-01-05':'2020-01-25']
fig,ax=plt.subplots(1,2,figsize=(15,4.5))
ax[0].plot(w.index,w['target_24h'],label='Actual (t+24h)'); ax[0].plot(w.index,w.pred,label='Random Forest'); ax[0].plot(w.index,w.persist,label='Persistence',alpha=.5)
ax[0].legend(); ax[0].set_title('Delhi: 24-hour-ahead AQI forecast, Jan 2020'); ax[0].tick_params(axis='x',rotation=30)
imp=pd.Series(rf.feature_importances_,index=FEAT).sort_values(ascending=False); imp.head(12)[::-1].plot(kind='barh',ax=ax[1],color='tab:green'); ax[1].set_title('Feature importance (24h-ahead)')
save('12_forecast_delhi_and_importance.png'); imp.head(8).round(3)

AQI           0.716
AQI_lag24     0.051
NO2           0.032
AQI_roll24    0.031
PM2.5         0.028
CO            0.018
O3            0.017
AQI_lag6      0.014
dtype: float64

In [4]:
# Error by forecast horizon bar chart
piv=fc.pivot(index='Model',columns='Horizon',values='MAE'); piv.plot(kind='bar',figsize=(8,4)); plt.ylabel('MAE (AQI points)'); plt.title('Forecast error by horizon (lower = better)'); plt.xticks(rotation=15); save('13_forecast_mae.png'); piv

Horizon,1h ahead,24h ahead
Model,,
Linear Regression,4.049,27.870
Persistence (baseline),2.733,23.838
Random Forest,2.982,25.622


## 2. Diwali effect (hourly resolution)
Hourly data shows *when* the spike happens - impossible in the daily file. Diwali dates (lunar calendar, listed manually as the guide says): 2015-11-11, 2016-10-30, 2017-10-19, 2018-11-07, 2019-10-27.
Spike ratio = peak hourly PM2.5 in the window [Diwali-1 day, Diwali+2 days] divided by the average PM2.5 in the baseline window 8-14 days before Diwali.

In [5]:
dw={2015:'2015-11-11',2016:'2016-10-30',2017:'2017-10-19',2018:'2018-11-07',2019:'2019-10-27'}
fig,ax=plt.subplots(1,2,figsize=(15,4.5)); out=[]
for city,a in zip(['Delhi','All 26 cities'],ax):
    for y,s in dw.items():
        s=pd.Timestamp(s); x=df if city=='All 26 cities' else df[df.City==city]
        win=x[(x.Datetime>=s-pd.Timedelta(days=3))&(x.Datetime<s+pd.Timedelta(days=4))].groupby('Datetime')['PM2.5'].mean()
        base=x[(x.Datetime>=s-pd.Timedelta(days=14))&(x.Datetime<s-pd.Timedelta(days=8))]['PM2.5'].mean()
        pk=win[(win.index>=s-pd.Timedelta(days=1))&(win.index<s+pd.Timedelta(days=2))]
        out.append([city,y,round(base,1),round(pk.max(),1) if len(pk) else np.nan,round(pk.max()/base,2) if len(pk) else np.nan])
        a.plot((win.index-s).total_seconds()/86400,win.values,label=str(y))
    a.axvline(0,color='k',ls='--',lw=.8); a.set_xlabel('Days relative to Diwali'); a.set_ylabel('PM2.5 (hourly mean)'); a.set_title(f'PM2.5 around Diwali - {city}'); a.legend()
save('14_diwali_hourly_pm25.png')
dwt=pd.DataFrame(out,columns=['Scope','Year','Baseline PM2.5 (8-14d before)','Peak PM2.5 (Diwali -1..+2d)','Spike ratio']); dwt.to_csv(RES+'city_hour_diwali_effect.csv',index=False); dwt

,Scope,Year,Baseline PM2.5 (8-14d before),Peak PM2.5 (Diwali -1..+2d),Spike ratio
0,Delhi,2015,178.8,514.9,2.88
1,Delhi,2016,146.7,759.3,5.17
2,Delhi,2017,90.4,797.5,8.82
3,Delhi,2018,198.1,682.2,3.44
4,Delhi,2019,123.5,597.9,4.84
5,All 26 cities,2015,112.2,429.2,3.83
6,All 26 cities,2016,91.4,460.8,5.04
7,All 26 cities,2017,57.9,427.8,7.39
8,All 26 cities,2018,93.0,361.8,3.89
9,All 26 cities,2019,56.1,364.6,6.50


## 3. COVID-19 lockdown effect
India's national lockdown began on **25 March 2020**. We compare 25 Mar - 30 Jun in 2020 against the same calendar window in 2018-2019 (same season, so seasonality is controlled), per city.

In [6]:
def win(x,y): return x[(x.Datetime>=f'{y}-03-25')&(x.Datetime<f'{y}-07-01')]
rows=[]
for c,g in df.groupby('City'):
    r={'City':c}
    for p in ['PM2.5','NO2','CO']:
        a=pd.concat([win(g,2018),win(g,2019)])[p].mean(); b=win(g,2020)[p].mean(); r[p]=100*(b-a)/a if pd.notna(a) and pd.notna(b) else np.nan
    rows.append(r)
lk=pd.DataFrame(rows).set_index('City').round(1); lk.to_csv(RES+'city_hour_lockdown_change_pct.csv')
fig,ax=plt.subplots(1,2,figsize=(15,6),gridspec_kw={'width_ratios':[1.1,1]})
lk['PM2.5'].dropna().sort_values().plot(kind='barh',ax=ax[0],color='tab:blue'); ax[0].axvline(0,color='k',lw=.8); ax[0].set_xlabel('% change vs 2018-19 (25 Mar-30 Jun)'); ax[0].set_title('PM2.5 change during lockdown window')
wk=df.set_index('Datetime')['PM2.5'].resample('W').mean(); wk.loc['2019-01-01':].plot(ax=ax[1]); ax[1].axvline(pd.Timestamp('2020-03-25'),color='r',ls='--'); ax[1].set_title('Weekly mean PM2.5 (all cities); red = lockdown start')
save('15_covid_lockdown.png'); print('Median change across cities (%):'); print(lk.median().round(1))

Median change across cities (%):
PM2.5   -40.9
NO2     -43.2
CO      -28.0
dtype: float64


## 4. Feature reduction: PCA vs raw pollutants
Same train/test split as notebook 1 (train 2015-2019, test 2020). We predict the current AQI from pollutants only, using (a) all raw pollutants, (b) the first *k* PCA components (fitted on training data only).

In [7]:
k=df.dropna(subset=['AQI']); tr=k[k.Year<=2019].sample(100000,random_state=RS); te=k[k.Year==2020]
imp=SimpleImputer(strategy='median').fit(tr[POLL]); sc=StandardScaler().fit(imp.transform(tr[POLL]))
Ztr,Zte=sc.transform(imp.transform(tr[POLL])),sc.transform(imp.transform(te[POLL]))
pca=PCA().fit(Ztr); cum=np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(6,4)); plt.plot(range(1,len(cum)+1),cum,marker='o'); plt.axhline(.9,color='r',ls='--'); plt.xlabel('Number of components'); plt.ylabel('Cumulative variance explained'); plt.title('PCA on pollutants'); save('16_pca_variance.png')
n90=int(np.argmax(cum>=.9))+1; print('Components for 90% variance:',n90,'of',len(POLL))
rows=[]
for name,(A,B) in {f'Raw pollutants ({len(POLL)})':(Ztr,Zte),f'PCA {n90} comps (90% var)':(pca.transform(Ztr)[:,:n90],pca.transform(Zte)[:,:n90]),'PCA 4 comps':(pca.transform(Ztr)[:,:4],pca.transform(Zte)[:,:4])}.items():
    rf=RandomForestRegressor(n_estimators=60,max_depth=18,min_samples_leaf=5,n_jobs=-1,random_state=RS).fit(A,tr.AQI); p=rf.predict(B)
    rows.append([name,mean_absolute_error(te.AQI,p),np.sqrt(mean_squared_error(te.AQI,p)),r2_score(te.AQI,p)])
pc=pd.DataFrame(rows,columns=['Features','MAE','RMSE','R2']).round(3); pc.to_csv(RES+'city_hour_pca_comparison.csv',index=False); pc

Components for 90% variance: 9 of 11


,Features,MAE,RMSE,R2
0,Raw pollutants (11),31.286,57.346,0.598
1,PCA 9 comps (90% var),37.200,70.761,0.389
2,PCA 4 comps,42.584,69.195,0.415


## 5. City comparison (hourly rhythm)

In [8]:
sel=[c for c in ['Delhi','Mumbai','Chennai','Kolkata','Bengaluru','Hyderabad','Lucknow','Ahmedabad'] if c in df.City.unique()]
hp=df[df.City.isin(sel)].pivot_table(index='City',columns='Hour',values='AQI',aggfunc='mean')
plt.figure(figsize=(12,4.5)); sns.heatmap(hp.div(hp.mean(axis=1),axis=0),cmap='RdYlBu_r',center=1,cbar_kws={'label':'AQI relative to city daily mean'}); plt.title('When in the day is each city\'s AQI above/below its own average?'); save('17_city_hourly_rhythm.png')
cs=df[df.City.isin(sel)].groupby('City').agg(mean_AQI=('AQI','mean'),p95_AQI=('AQI',lambda s:s.quantile(.95)),mean_PM25=('PM2.5','mean')).round(1).sort_values('mean_AQI',ascending=False)
cs.to_csv(RES+'city_hour_city_comparison.csv'); cs

,mean_AQI,p95_AQI,mean_PM25
City,,,
Ahmedabad,455.0,1445.3,67.3
Delhi,260.2,461.0,117.7
Lucknow,218.4,410.0,110.0
Kolkata,140.7,360.0,64.7
Chennai,114.2,226.0,50.2
Hyderabad,109.3,198.0,47.1
Mumbai,106.1,204.0,35.5
Bengaluru,94.4,191.0,35.6


## 6. SQLite export (matches the team's `aqi.db` approach)

In [9]:
con=sqlite3.connect('../data/Cleaned/city_hour.db') if os.path.isdir('../data/Cleaned') else sqlite3.connect('../data/processed/city_hour.db')
df[['City','Datetime','AQI','AQI_Bucket']+POLL].to_sql('city_hour_clean',con,if_exists='replace',index=False)
print(pd.read_sql("SELECT City, ROUND(AVG(AQI),1) avg_aqi FROM city_hour_clean GROUP BY City ORDER BY avg_aqi DESC LIMIT 5",con)); con.close()

        City  avg_aqi
0  Ahmedabad    455.0
1      Delhi    260.2
2      Patna    236.7
3   Gurugram    223.6
4    Lucknow    218.4
